# Knowledge tracing: the LMS mastery rule against standard baselines

This notebook predicts a student's **next answer** on ASSISTments 2009 (skill-builder, corrected version) and compares:

| Model | What it is |
| --- | --- |
| **LMS mastery rule** | The update rule from the Adaptive AI-Powered LMS (`app/mastery/core.py`), with its default numbers. Nothing is fitted. |
| per-skill mean | Predicts each skill's average correctness in the training students. |
| global mean | A constant. Its AUC is always 0.5. |
| **pyBKT** | A standard Bayesian Knowledge Tracing model fitted for each skill. |

Metric: **AUC** over all test predictions. Students are split into folds, so no test student is seen while fitting.

**How to run it (first time on Kaggle)**
1. Make a Kaggle account and verify your phone number (Kaggle needs this to turn on the internet in notebooks).
2. Upload `skill_builder_data.csv` as a **private dataset**: *Datasets -> New Dataset -> drag the file in -> Create*.
3. Create a notebook: *Code -> New Notebook*. Use *File -> Import Notebook* to load this file.
4. In the right-hand panel: *Input -> Add Input* and add your dataset. Under *Session options* turn **Internet on** (needed once, to install pyBKT).
5. Press *Run All*.

In [ ]:
import os, re, subprocess, sys, glob

try:
    import pyBKT  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyBKT"])

import numpy as np
import pandas as pd
import sklearn.metrics as skm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

FOLDS = 5                 # use 3 for a quicker run
MAX_STUDENTS = None       # for example 1500 to try it quickly on a sample

## 1. Find and load the data
The notebook looks for a file named like `skill_builder_data*.csv` among your inputs. Rows without a skill are dropped,
and each student's answers are put in time order (`order_id`).

In [ ]:
path = os.environ.get("KT_DATA") or next(iter(sorted(
    glob.glob("/kaggle/input/**/skill_builder_data*.csv", recursive=True))), None)
if path is None:
    raise FileNotFoundError("Add your skill_builder_data.csv dataset under Input (right-hand panel) and run again.")

raw = pd.read_csv(path, encoding="latin-1", low_memory=False).dropna(subset=["skill_id"])
data = pd.DataFrame({
    "user_id": raw["user_id"], "skill_name": raw["skill_id"].astype(str),
    "correct": raw["correct"].astype(int), "order_id": raw["order_id"],
}).sort_values("order_id").reset_index(drop=True)
if MAX_STUDENTS:
    keep = pd.Series(data["user_id"].unique()).sample(MAX_STUDENTS, random_state=0)
    data = data[data["user_id"].isin(keep)].reset_index(drop=True)
print(f"{len(data)} answers, {data.user_id.nunique()} students, {data.skill_name.nunique()} skills, "
      f"{data.correct.mean():.3f} correct")

## 2. The LMS mastery rule
Copied from `backend/app/mastery/core.py` of the LMS repository: the Bayes update, and the default slip and guess for a
graded program (`all_pass`). Every student starts at 0.5 on every skill (the prior the LMS uses for its concepts). The
prediction before each answer is `p * (1 - slip) + (1 - p) * guess`. There is no learning step, as in the LMS.

In [ ]:
EPS = 1e-6
SLIP, GUESS = 0.10, 0.20    # DEFAULT_PARAMS["all_pass"] in app/mastery/core.py
PRIOR = 0.5


def clamp(p):
    return min(1.0 - EPS, max(EPS, p))


def bayes_update(p, slip, guess, correct):
    if correct:
        numerator = p * (1 - slip)
        denominator = numerator + (1 - p) * guess
    else:
        numerator = p * slip
        denominator = numerator + (1 - p) * (1 - guess)
    return clamp(numerator / denominator)


def predict_lms(test):
    out = np.empty(len(test))
    state = {}
    for position, (user, skill, correct) in enumerate(zip(test["user_id"], test["skill_name"], test["correct"])):
        p = state.get((user, skill), PRIOR)
        out[position] = p * (1 - SLIP) + (1 - p) * GUESS
        state[(user, skill)] = bayes_update(p, SLIP, GUESS, bool(correct))
    return out

## 3. pyBKT
pyBKT probes scikit-learn at import and stumbles on newer versions, and one line of its fitting code breaks with newer
numpy. The two small fixes below change nothing about the model; they only keep it from crashing. Each skill gets its own model, fitted in one
process (`parallel=False`) to stay inside Kaggle's memory.

In [ ]:
def tolerant(function):
    def call(*args, **kwargs):
        try:
            return function(*args, **kwargs)
        except AttributeError as exc:
            raise TypeError(str(exc)) from None
    return call


for module in (skm._regression, skm._classification):
    for name in dir(module):
        if re.search("_loss$|_score$|_error$", name):
            setattr(module, name, tolerant(getattr(module, name)))

import importlib.util

fit_file = os.path.join(os.path.dirname(importlib.util.find_spec("pyBKT").origin), "fit", "EM_fit.py")
text = open(fit_file, encoding="utf8").read()
old = "log_likelihoods[i][0] = result['total_loglike']"
if old in text:
    open(fit_file, "w", encoding="utf8").write(text.replace(
        old, "log_likelihoods[i][0] = float(__import__('numpy').asarray(result['total_loglike']).ravel()[0])"))

from pyBKT.models import Model


def predict_pybkt(train, test, fallback):
    # One model per skill. Fitting all skills in a single call gave AUC about 0.51 in testing; this gives about 0.70.
    parts = []
    for skill, rows in test.groupby("skill_name"):
        fitted = train[train["skill_name"] == skill]
        if fitted["correct"].nunique() < 2:
            continue
        model = Model(parallel=False, num_fits=1, seed=0)
        model.fit(data=fitted)
        parts.append(model.predict(data=rows))
    got = pd.concat(parts).set_index("order_id")["correct_predictions"]
    values = test["order_id"].map(got)
    return values.fillna(fallback).to_numpy(), int(values.isna().sum())

## 4. Run the comparison

In [ ]:
scores = {"lms": [], "global": [], "skill": [], "pybkt": []}
unmatched = 0
for fold, (tr, te) in enumerate(GroupKFold(n_splits=FOLDS).split(data, groups=data["user_id"]), 1):
    train, test = data.iloc[tr], data.iloc[te].reset_index(drop=True)
    base = float(train["correct"].mean())
    by_skill = train.groupby("skill_name")["correct"].mean()
    truth = test["correct"].to_numpy()
    scores["lms"].append(roc_auc_score(truth, predict_lms(test)))
    scores["global"].append(0.5)
    scores["skill"].append(roc_auc_score(truth, test["skill_name"].map(by_skill).fillna(base).to_numpy()))
    values, missing = predict_pybkt(train, test, base)
    unmatched += missing
    scores["pybkt"].append(roc_auc_score(truth, values))
    print(f"fold {fold}: " + ", ".join(f"{k} {v[-1]:.4f}" for k, v in scores.items()), flush=True)

labels = {"lms": "LMS mastery rule (nothing fitted)", "global": "global mean (constant)",
          "skill": "per-skill mean", "pybkt": "pyBKT, standard BKT per skill"}
table = pd.DataFrame({
    "Model": [labels[k] for k in scores],
    "Mean AUC": [round(float(np.mean(v)), 4) for v in scores.values()],
    "Std over folds": [round(float(np.std(v)), 4) for v in scores.values()],
})
print(table.to_string(index=False))
if unmatched:
    print(f"pyBKT gave no prediction for {unmatched} test answers; they received the training mean.")

## How to read the result
* Higher AUC is better; 0.5 is no better than a coin.
* Report the table exactly as printed, with the date and the number of folds. A different fold split gives slightly
  different numbers, so quote the standard deviation too.
* Published results for this dataset use their own splits and preprocessing. Compare with them only after checking that
  they used the same cleaned data (the original ASSISTments 2009 file has duplicate records) and the same prediction task.
* This tests the **mastery rule only**. It does not test the explanations or the adaptive placement.